In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
tfidf = TfidfVectorizer(stop_words='english')

movies['genres'] = movies['genres'].fillna('')

tfidf_matrix = tfidf.fit_transform(movies['genres'])

In [5]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [6]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

In [7]:
def recommend_movies(title, cosine_sim=cosine_sim):
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:6]  # Top 5
    
    movie_indices = [i[0] for i in sim_scores]
    
    return movies['title'].iloc[movie_indices]

In [8]:
recommend_movies("Toy Story (1995)")

1706                                       Antz (1998)
2355                                Toy Story 2 (1999)
2809    Adventures of Rocky and Bullwinkle, The (2000)
3000                  Emperor's New Groove, The (2000)
3568                             Monsters, Inc. (2001)
Name: title, dtype: object

In [9]:
movie_ratings = ratings.pivot_table(index='userId', columns='movieId', values='rating')

movie_ratings = movie_ratings.fillna(0)

In [10]:
import joblib

joblib.dump(cosine_sim, "cosine_similarity.pkl")
joblib.dump(movies, "movies.pkl")

['movies.pkl']